# GridPulse BR — Regulatory Limits Silver

## Objective

Transform the ANEEL regulatory continuity limits source into a trusted Silver
dataset for downstream comparison with actual DEC and FEC performance.

## Silver responsibilities

- Normalize regulatory limit values
- Normalize CNPJ identifiers
- Standardize distributor attributes
- Apply the validated regulatory-limit business grain
- Remove known duplicate regulatory observations
- Validate uniqueness and domain consistency
- Persist a trusted Delta table

In [0]:
from pyspark.sql import functions as F

SOURCE_FILE = (
    "/Volumes/workspace/gridpulse/landing/aneel/"
    "continuity_limits/indicadores-continuuidade-coletivos-limite.csv"
)

TARGET_TABLE = "workspace.gridpulse.silver_aneel_regulatory_limits"

LIMIT_KEY = [
    "IdeConjUndConsumidoras",
    "SigIndicador",
    "AnoLimiteQualidade"
]

print(f"Source file  : {SOURCE_FILE}")
print(f"Target table : {TARGET_TABLE}")
print(f"Business key : {LIMIT_KEY}")

In [0]:
SOURCE_FILE = (
    "/Volumes/workspace/gridpulse/landing/aneel/"
    "continuity_limits/indicadores-continuidade-coletivos-limite.csv"
)

## 1. Read raw regulatory limits

In [0]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(SOURCE_FILE)
)

raw_row_count = df_raw.count()

print(f"Raw rows    : {raw_row_count:,}")
print(f"Raw columns : {len(df_raw.columns)}")

df_raw.printSchema()

## 2. Standardize regulatory limits

In [0]:
df_standardized = (
    df_raw
    .withColumn(
        "NumCNPJ",
        F.lpad(
            F.regexp_replace(F.col("NumCNPJ"), r"\D", ""),
            14,
            "0"
        )
    )
    .withColumn(
        "SigAgente",
        F.trim(F.col("SigAgente"))
    )
    .withColumn(
        "VlrLimite",
        F.regexp_replace(
            F.col("VlrLimite"),
            ",",
            "."
        ).cast("double")
    )
    .withColumn(
        "AnoLimiteQualidade",
        F.col("AnoLimiteQualidade").cast("int")
    )
    .withColumn(
        "IdeConjUndConsumidoras",
        F.col("IdeConjUndConsumidoras").cast("long")
    )
)

## 3. Deduplicate validated regulatory observations

The source profiling identified 12 business-key collisions with identical
regulatory limit values and different distributor ownership attributes.

For regulatory comparison purposes, the business grain is defined by
consumer-unit set, indicator and quality-limit year.

In [0]:
from pyspark.sql.window import Window

dedup_window = (
    Window
    .partitionBy(*LIMIT_KEY)
    .orderBy(
        F.col("DatGeracaoConjuntoDados").desc(),
        F.col("SigAgente").asc(),
        F.col("NumCNPJ").asc()
    )
)

df_silver_limits = (
    df_standardized
    .withColumn(
        "_row_number",
        F.row_number().over(dedup_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

silver_row_count = df_silver_limits.count()

print(f"Raw rows       : {raw_row_count:,}")
print(f"Silver rows    : {silver_row_count:,}")
print(f"Rows removed   : {raw_row_count - silver_row_count:,}")

## 4. Validate Silver regulatory grain

In [0]:
duplicate_keys = (
    df_silver_limits
    .groupBy(*LIMIT_KEY)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_key_count = duplicate_keys.count()

print(f"Remaining duplicate regulatory keys : {duplicate_key_count:,}")

assert duplicate_key_count == 0, (
    f"Regulatory grain validation failed: "
    f"{duplicate_key_count:,} duplicate keys remain."
)

print("Regulatory grain validation passed.")

In [0]:
domain_profile = (
    df_silver_limits
    .agg(
        F.min("AnoLimiteQualidade").alias("min_year"),
        F.max("AnoLimiteQualidade").alias("max_year"),
        F.min("VlrLimite").alias("min_limit"),
        F.max("VlrLimite").alias("max_limit"),
        F.countDistinct("SigIndicador").alias("indicator_count")
    )
)

display(domain_profile)

## 5. Persist regulatory limits Silver table

In [0]:
(
    df_silver_limits.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Silver regulatory limits table created successfully: {TARGET_TABLE}")

In [0]:
df_persisted = spark.table(TARGET_TABLE)

persisted_count = df_persisted.count()

print(f"Expected rows : {silver_row_count:,}")
print(f"Actual rows   : {persisted_count:,}")

assert persisted_count == silver_row_count

print("Regulatory limits Silver reconciliation passed.")